In [2]:
import os
import uuid
from dotenv import load_dotenv
from langgraph.store.memory import InMemoryStore
from langchain_core.rate_limiters import InMemoryRateLimiter
from langchain_openai import AzureOpenAIEmbeddings, AzureChatOpenAI
from typing import Annotated
from typing_extensions import TypedDict

from langchain_core.runnables import RunnableConfig
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.checkpoint.memory import MemorySaver
from langgraph.store.base import BaseStore

In [3]:
load_dotenv()

True

# Setting the Required Tools

In [4]:
in_memory_store = InMemoryStore(
    index={
        "embed": AzureOpenAIEmbeddings(
            api_key=os.getenv("AZURE_OPENAI_API_KEY_4O_MINI"),    
            azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT_4O_MINI"),
            api_version=os.getenv("AZURE_OPENAI_API_VERSION_EMBEDDING"),
            azure_deployment=os.getenv("AZURE_EMBEDDING_ENGINE"),
            model=os.getenv("AZURE_EMBEDDING_MODEL_NAME"),
        ),
        "dims": 1536,
    }
)

In [5]:


# LLMs

rate_limiter = InMemoryRateLimiter(
    requests_per_second=4,
    check_every_n_seconds=0.1,
    max_bucket_size=10,  # Controls the maximum burst size.
)

model = AzureChatOpenAI(
                api_key=os.getenv("AZURE_OPENAI_API_KEY_4O_MINI"),    
                azure_endpoint=os.getenv("AZURE_OPENAI_ENDPOINT_4O_MINI"),
                api_version=os.getenv("AZURE_OPENAI_API_VERSION_4O_MINI"),
                azure_deployment=os.getenv("AZURE_ENGINE_4O_MINI"),
                model=os.getenv("AZURE_MODEL_NAME_4O_MINI"),
                temperature=0,
                rate_limiter=rate_limiter
            )

/tmp/ipykernel_414359/2512488115.py:3: LangChainBetaWarning: Introduced in 0.2.24. API subject to change.
  rate_limiter = InMemoryRateLimiter(


## Function that will keep calling the AI Model and maintaining Chat History

In [6]:
def call_model(state: MessagesState, config: RunnableConfig, *, store: BaseStore):
    
    # getting unique user id
    user_id = config["configurable"]["user_id"]
    
    # creating a separate namespace in the vector store for each user id. This will help us in maintaining long term conversations of user in VectorStore
    namespace = ("memories", user_id)
    
    # getting related messages
    memories = store.search(namespace, query=str(state["messages"][-1].content))
    
    info = "\n".join([d.value["data"] for d in memories])
    
    # passing related messages into prompt
    system_msg = f"You are a helpful assistant talking to the user. User info: {info}"

    # Store new memories if the user asks the model to remember
    last_message = state["messages"][-1]
    
    if "remember" in last_message.content.lower():
        memory = last_message.content
        store.put(namespace, str(uuid.uuid4()), {"data": memory})

    response = model.invoke(
        [{"type": "system", "content": system_msg}] + state["messages"]
    )
    return {"messages": response}


In [11]:
builder = StateGraph(MessagesState)
builder.add_node("call_model", call_model)
builder.add_edge(START, "call_model")


In [12]:
graph = builder.compile(checkpointer=MemorySaver(), store=in_memory_store)

# Setting Configurations to Run Graph

In [7]:
config = {"configurable": {"thread_id": "1", "user_id": "1"}}

In [9]:

input_message = {"type": "user", "content": "Hi! Remember: my name is Bob"}

In [13]:

for chunk in graph.stream({"messages": [input_message]}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

Hi! Remember: my name is Bob
================================== Ai Message ==================================

Hi Bob! It's great to chat with you. How can I assist you today?


In [14]:
input_message = {"type": "user", "content": "What is Neural Network, explain it to me like i am 15 years old"}

In [15]:
for chunk in graph.stream({"messages": [input_message]}, config, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

What is Neural Network, explain it to me like i am 15 years old
================================== Ai Message ==================================

Sure, Bob! Imagine your brain is like a big network of tiny lights, where each light can turn on or off. These lights represent neurons, which are the building blocks of your brain. A neural network is a computer system that tries to mimic how our brains work, using a similar idea of interconnected "neurons."

Here’s a simple breakdown:

1. **Neurons**: In a neural network, there are many small units called neurons. Each neuron takes in some information (like numbers), processes it, and then passes it on to other neurons.

2. **Layers**: Neurons are organized into layers. The first layer takes in the input (like an image or a piece of text), the middle layers do the processing, and the last layer gives the output (like identifying what’s in the image).

3. **Con

### Checking if We create another user, will it interfere with the current graph state

In [16]:
config2 = {"configurable": {"thread_id": "10", "user_id": "10"}}

In [17]:
input_message2 = {"type": "user", "content": "Hi, Do you know my name?"}

In [18]:
for chunk in graph.stream({"messages": [input_message2]}, config2, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

Hi, Do you know my name?
================================== Ai Message ==================================

No, I don't know your name. But I'm here to help you with anything you need! What would you like to talk about?


In [19]:
input_message2 = {"type": "user", "content": "Who is Bob then?"}
for chunk in graph.stream({"messages": [input_message2]}, config2, stream_mode="values"):
    chunk["messages"][-1].pretty_print()

================================ Human Message =================================

Who is Bob then?
================================== Ai Message ==================================

"Bob" could refer to many people, as it's a common name. If you have a specific Bob in mind, such as a public figure, a character from a book or movie, or someone in your life, please provide more context, and I'll do my best to help!
